# Phase 2: Descriptive Analysis
This phase establishes the baseline health of the patients. The clinical gold standard for T1DM analysis is Time in Range (TIR).

## Key Columns & Importance:
* glucose: The primary target. Clinicians look at the percentage of time spent in the target range (70–180 mg/dL), hypoglycemia (< 70), and hyperglycemia (> 180).
* carb_input & bolus_volume_delivered: The primary drivers of upward and downward glucose swings.

The output of the descriptive_metrics(df) function—specifically the daily_stats variable returned at the end—is a Pandas Series containing the average daily totals for the patient's carbohydrates, fast-acting insulin (bolus), and step count.

Because it calculates the baseline behavior of the patient, it is highly useful for two specific deliverables in your hackathon: populating your dashboard UI and establishing thresholds for your prescriptive analysis.

In [1]:
import import_ipynb
from Data_Preprocessing_Cleaning import read_and_preprocess_data

In [2]:
def descriptive_metrics(df):
    total_readings = len(df)
    
    # Calculate Clinical Time in Range (TIR)
    tir = len(df[(df['glucose'] >= 70) & (df['glucose'] <= 180)]) / total_readings * 100
    hypo = len(df[df['glucose'] < 70]) / total_readings * 100
    hyper = len(df[df['glucose'] > 180]) / total_readings * 100
    
    print(f"Time in Range (70-180 mg/dL): {tir:.1f}%")
    print(f"Time in Hypoglycemia (<70 mg/dL): {hypo:.1f}%")
    print(f"Time in Hyperglycemia (>180 mg/dL): {hyper:.1f}%")
    
    # Basic aggregations
    daily_stats = df.resample('D').agg({
        'carb_input': 'sum',
        'bolus_volume_delivered': 'sum',
        'steps': 'sum'
    }).mean()
    
    return daily_stats

In [3]:
cleaned_df = read_and_preprocess_data('/Users/jputha177@cable.comcast.com/Downloads/HUPA0002P.csv')


In [4]:
# 1. Run the function and store the returned pandas Series
patient_baselines = descriptive_metrics(cleaned_df)

# 2. Extract the individual values
avg_carbs = patient_baselines['carb_input']
avg_insulin = patient_baselines['bolus_volume_delivered']
avg_steps = patient_baselines['steps']

print(f"This patient eats an average of {avg_carbs:.0f}g of carbs per day.")

Time in Range (70-180 mg/dL): 61.5%
Time in Hypoglycemia (<70 mg/dL): 23.9%
Time in Hyperglycemia (>180 mg/dL): 14.6%
This patient eats an average of 53g of carbs per day.


In [ ]:
# Calculate total steps over a rolling 24-hour window
df['rolling_24h_steps'] = df['steps'].rolling('1D').sum()

# Define a "High Activity Day" marker as any time their rolling steps 
# exceed their personal daily average by 20%
high_activity_threshold = patient_baselines['steps'] * 1.2

# Create the categorical marker column
df['activity_marker'] = df['rolling_24h_steps'].apply(
    lambda x: 'High Activity' if x > high_activity_threshold else 'Normal/Low Activity'
)

# Now you can filter the data to see how insulin sensitivity changes during 'High Activity'
high_activity_data = df[df['activity_marker'] == 'High Activity']

In [43]:
df['rolling_24h_steps']

time
2018-06-13 22:45:00        0.0
2018-06-13 22:50:00        0.0
2018-06-13 22:55:00        0.0
2018-06-13 23:00:00        0.0
2018-06-13 23:05:00        0.0
                        ...   
2018-06-24 23:25:00    11123.0
2018-06-24 23:30:00    11067.0
2018-06-24 23:35:00    10927.0
2018-06-24 23:40:00    10444.0
2018-06-24 23:45:00     9965.0
Freq: 5min, Name: rolling_24h_steps, Length: 3181, dtype: float64